# Expert analysis

In [3]:
# first we load the data.
# All data is provided in a single excel report sheet which is the single source of trust here
from pathlib import Path
import pandas as pd

ROOT = Path("/local/joeldag/repos/moe-study")
REPORT = ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/expert_pipeline_report.xlsx"
xls = pd.ExcelFile(REPORT)
raw = pd.read_excel(REPORT, sheet_name="benchmark_raw")
source = pd.read_excel(REPORT, sheet_name="source_manifest")
audit = pd.read_excel(REPORT, sheet_name="audit_status")

primary = raw[
  ((raw.benchmark == "Belebele") & (raw.metric == "acc_norm")) |
  ((raw.benchmark == "FLORES") & (raw.metric.astype(str).str.startswith("bits_per_byte "))) |
  ((raw.benchmark == "SIB200") & (raw.metric == "acc")) |
  ((raw.benchmark == "Taxi1500") & (raw.metric == "acc"))
].copy() # primary metric we use in our analysis
primary = primary[~primary["is_baseline"].astype(str).str.lower().isin(["true", "1", "yes"])] # remove baseline
primary.head()

,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline
0,adamoe_1_r2_e16_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,False
1,adamoe_1_r2_e16_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.588889,False
2,adamoe_1_r2_e16_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.594444,False
3,adamoe_1_r2_e16_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.597778,False
4,adamoe_1_r2_e2_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.577778,False


In [4]:
# here we prepare our df for analysis and normalize the scores
import re
import numpy as np

# helper, extracts the expert count from a run_id, e.g. moelora_r2_e16_k8 -> 16, hydralora_r2_heads32 -> 32 mtllora_r2_nup8 -> 8, cola_r2_a3_b4 -> 4
def parse_count(run_id):
  for pat in [r"_e(\d+)(?:_|$)", r"_heads(\d+)$", r"_nup(\d+)$", r"_b(\d+)$"]:
      m = re.search(pat, str(run_id))
      if m:
          return int(m.group(1))
  return np.nan

metrics = primary.copy()
len(metrics)
metrics["direction"] = metrics.metric.astype(str).str.extract(r"(en->x|x->en)", expand=False).fillna("") # extract metrics direction
metrics["score_family"] = np.where(metrics.benchmark.eq("FLORES"), "FLORES " + metrics.direction, metrics.benchmark) #
metrics["component_count"] = metrics.run_id.map(parse_count) # e.g. for 8 experts its 8
metrics["top_k"] = metrics.run_id.astype(str).str.extract(r"_k(\d+)(?:_|$)", expand=False).astype(float)
metrics["axis_label"] = metrics.kind.map({"hydralora": "heads", "mtllora": "up_projections", "cola": "cola_B"}).fillna("experts")
metrics["value"] = pd.to_numeric(metrics.value, errors="coerce") # ensure numeric values, NaN else

# we normalize metrics using min - max normalization. within each benchmark family (belebele, flores, taxi, sib) each result gets a value between 0-1
g = metrics.groupby(["score_family", "task", "metric"]).value
lo, hi = g.transform("min"), g.transform("max")
metrics["normalized_score"] = np.where(metrics.better.eq("lower"), (hi - metrics.value) / (hi - lo), (metrics.value - lo) / (hi - lo))
metrics.loc[(hi - lo).eq(0), "normalized_score"] = 1.0 # edge case if all have same value

metrics = metrics.dropna(subset=["value", "component_count"]).copy()
metrics["component_count"] = metrics.component_count.astype(int)
metrics.head()

,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline,direction,score_family,component_count,top_k,axis_label,normalized_score
0,adamoe_1_r2_e16_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,False,,Belebele,16,1.0,experts,0.946779
1,adamoe_1_r2_e16_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.588889,False,,Belebele,16,2.0,experts,0.918768
2,adamoe_1_r2_e16_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.594444,False,,Belebele,16,4.0,experts,0.932773
3,adamoe_1_r2_e16_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.597778,False,,Belebele,16,8.0,experts,0.941176
4,adamoe_1_r2_e2_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.577778,False,,Belebele,2,1.0,experts,0.890756


In [5]:
# we have 68 training runs * (9xbelebele metrics + 18 FLORES + 9 SIB + 6 TAXI) = 2856 metrics we should have for all approaches x all datasets x all languages
# our df should have that many entries with each entry having the exact metric results as well as the normalize score and metadata of the run
len(metrics)

2856

In [6]:
# last sanity checks
print("rows:", len(metrics))
print("runs:", metrics.run_id.nunique())
print("score families:", metrics.score_family.unique())

# should be 0 missing after prep
print(metrics[["value", "normalized_score", "component_count"]].isna().sum())

# coverage per score family
metrics.groupby("score_family").agg(
  rows=("run_id", "count"),
  runs=("run_id", "nunique"),
  tasks=("task", "nunique"),
  languages=("language", "nunique"))

rows: 2856
runs: 68
score families: ['Belebele' 'FLORES en->x' 'FLORES x->en' 'SIB200' 'Taxi1500']
value               0
normalized_score    0
component_count     0
dtype: int64


,rows,runs,tasks,languages
score_family,,,,
Belebele,612,68,9,9
FLORES en->x,612,68,9,9
FLORES x->en,612,68,9,9
SIB200,612,68,9,9
Taxi1500,408,68,6,6


# Which expert count wins per benchmark and is the result significant?

In [10]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np


# We use Holm correction here for confirmatory post-hoc comparisons, because our goal is to make conservative winner claims while controlling
# the family-wise error rate: the chance of making at least one false claim.
#
# In contrast, pipeline_tools/reporting/significance.py uses Benjamini-Hochberg FDR correction, which is better suited for exploratory screening across many tests where we want to control the expected share of false discoveries.
# https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method

COUNTS = [2, 4, 8, 16, 32]

def holm(pvals):
  return multipletests(pd.Series(pvals, dtype=float), method="holm")[1]

def count_pivot(df, group_cols):
  return (
      df.groupby(group_cols + ["component_count"])["normalized_score"]
      .mean()
      .unstack("component_count")
      .reindex(columns=COUNTS)
      .dropna()
  )

def friedman_count_overall(df, label):
  pivot = count_pivot(df, ["score_family", "task", "language", "metric"])

  stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
  avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
  top3 = avg_ranks.index[:3].tolist()

  return {
      "analysis": label,
      "pivot": pivot,
      "top3_counts": top3,
      "friedman_p_raw": p,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in COUNTS},
  }

def wilcoxon_adjacent_top3(result):
  pivot = result["pivot"]
  top3 = result["top3_counts"]

  rows = []
  # we compare if 1st and 2nd best and 2nd and 3rd best are significant using Wilconxon signed rank
  for a, b in [(top3[0], top3[1]), (top3[1], top3[2])]:
      w = wilcoxon(pivot[a], pivot[b], zero_method="pratt", alternative="greater")
      rows.append({
          "analysis": result["analysis"],
          "a": a,
          "b": b,
          "mean_a": pivot[a].mean(),
          "mean_b": pivot[b].mean(),
          "median_delta_a_minus_b": (pivot[a] - pivot[b]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out

def friedman_count_by_benchmark(df, label):
  rows = []

  # here we analyze per benchmark family and use fridmann test to derive which expert count performs best per family
  for score_family, g in df.groupby("score_family"):
      pivot = count_pivot(g, ["task", "language", "metric"])

      stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
      avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()

      rows.append({
          "analysis": label,
          "score_family": score_family,
          "winner_count": avg_ranks.index[0],
          "n_units": len(pivot),
          "p_raw": p,
          "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
          **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in COUNTS},
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out

# Overall: top 3 counts across all benchmark families
# we check with and without cola since cola seems to skew resutls for higher expert count
overall_all = friedman_count_overall(metrics, "with_cola")
overall_no_cola = friedman_count_overall(metrics[metrics.kind.ne("cola")], "no_cola")

overall_summary = pd.DataFrame([
  {k: v for k, v in overall_all.items() if k != "pivot"},
  {k: v for k, v in overall_no_cola.items() if k != "pivot"},
])
overall_summary["friedman_p_holm"] = holm(overall_summary["friedman_p_raw"]) # perform holm correction on friedman raw results
overall_summary["significant"] = overall_summary["friedman_p_holm"] < 0.05   # check if significant by 0.05

print("Overall top 3 counts")
print("This indicates which top3 expert counts we derive using cola and excluding cola")
display(overall_summary)

print("Now we look closer into the top3 beest experts coutns and check whether the difference between 1st and 2nd best and 2nd abd 3rd best is significant")
print("Wilcoxon: adjacent top counts, with CoLA")
display(wilcoxon_adjacent_top3(overall_all))

print("Wilcoxon: adjacent top counts, without CoLA")
display(wilcoxon_adjacent_top3(overall_no_cola))

# Per benchmark family: best count and significance
print("Now we check per benchmark family which expert count is the best. We only use the best corrected friedman result here. No additional Wilconxon")
print("Per benchmark family, with CoLA")
display(friedman_count_by_benchmark(metrics, "with_cola"))

print("Per benchmark family, without CoLA")
display(friedman_count_by_benchmark(metrics[metrics.kind.ne("cola")], "no_cola"))

Overall top 3 counts
This indicates which top3 expert counts we derive using cola and excluding cola


,analysis,top3_counts,friedman_p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,friedman_p_holm,significant
0,with_cola,"[2, 4, 8]",3.073507e-18,0.525397,1.738095,2.047619,3.071429,4.547619,3.595238,6.147015e-18,True
1,no_cola,"[32, 2, 4]",8.742769e-15,0.428345,2.547619,2.738095,4.142857,3.928571,1.642857,8.742769e-15,True


Now we look closer into the top3 beest experts coutns and check whether the difference between 1st and 2nd best and 2nd abd 3rd best is significant
Wilcoxon: adjacent top counts, with CoLA


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,with_cola,2,4,0.890187,0.886199,0.000778,42,6.564972e-02,0.065650,False
1,with_cola,4,8,0.886199,0.874452,0.005422,42,8.947236e-07,0.000002,True


Wilcoxon: adjacent top counts, without CoLA


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,no_cola,32,2,0.903683,0.884799,0.001928,42,0.000002,0.000004,True
1,no_cola,2,4,0.884799,0.882127,0.000865,42,0.083413,0.083413,False


Now we check per benchmark family which expert count is the best. We only use the best corrected friedman result here. No additional Wilconxon
Per benchmark family, with CoLA


,analysis,score_family,winner_count,n_units,p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,p_holm,significant
0,with_cola,Belebele,2,9,1.110009e-06,0.920988,1.111111,2.555556,4.000000,5.000000,2.333333,0.000003,True
1,with_cola,FLORES en->x,2,9,6.709404e-07,0.950617,1.000000,2.000000,3.000000,4.555556,4.444444,0.000003,True
2,with_cola,FLORES x->en,2,9,6.709404e-07,0.950617,1.000000,2.000000,3.000000,4.555556,4.444444,0.000003,True
3,with_cola,SIB200,4,9,5.431643e-06,0.827160,2.888889,1.000000,2.444444,3.888889,4.777778,0.000011,True
4,with_cola,Taxi1500,32,6,2.681437e-03,0.677778,3.166667,3.000000,2.833333,4.833333,1.166667,0.002681,True


Per benchmark family, without CoLA


,analysis,score_family,winner_count,n_units,p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,p_holm,significant
0,no_cola,Belebele,32,9,5.915120e-07,0.958025,2.000000,3.111111,4.888889,4.000000,1.000000,0.000003,True
1,no_cola,FLORES en->x,32,9,9.386181e-07,0.930864,1.555556,3.000000,4.111111,4.888889,1.444444,0.000004,True
2,no_cola,FLORES x->en,32,9,9.386181e-07,0.930864,1.777778,3.000000,4.222222,4.777778,1.222222,0.000004,True
3,no_cola,SIB200,4,9,4.079662e-04,0.567901,4.222222,1.222222,3.777778,2.444444,3.333333,0.000816,True
4,no_cola,Taxi1500,32,6,1.638682e-02,0.505556,3.500000,3.666667,3.500000,3.333333,1.000000,0.016387,True


### Interpretation

All results are based on the normalized task-level scores.
For each task/language/metric, we first min-max normalize scores across runs, then aggregate these normalized
scores across benchmark families.

#### Overall expert/component-count ranking

If CoLA is included:

- Top 3 counts are: `2`, `4`, `8`
- The difference between rank 1 and rank 2 (`2` vs `4`) is **not significant**
- This result is likely influenced by CoLA collapsing at higher counts (`16`, `32`)

If CoLA is excluded:

- Top 3 counts are: `32`, `2`, `4`
- The difference between rank 1 and rank 2 (`32` vs `2`) is **significant**
- This suggests that the low-count result is not stable once the CoLA failure mode is removed

#### Per-benchmark Friedman winners

Without CoLA:

| Benchmark family | Best count |
|---|---:|
| Belebele | 32 |
| FLORES en->x | 32 |
| FLORES x->en | 32 |
| SIB200 | 4 |
| Taxi1500 | 32 |

With CoLA:

| Benchmark family | Best count |
|---|---:|
| Belebele | 2 |
| FLORES en->x | 2 |
| FLORES x->en | 2 |
| SIB200 | 4 |
| Taxi1500 | 32 |

# Now we look closer into all approaches that have a topk value: Which expert count x topk works the best?

In [17]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# We only look at approaches that have top-k now
# We ignore the specific method family (s.a adamoe_1/mola, hmora, moelora) and treat each expert_count x top_k pair as one combo
# Example: e16 k2 combines all runs with 16 experts/components and top-k 2.

def holm(pvals):
    return multipletests(pd.Series(pvals, dtype=float), method="holm")[1]


def combo_pivot(df, group_cols):
    return (
      df.groupby(group_cols + ["combo"])["normalized_score"]
      .mean()
      .unstack("combo")
      .dropna()
    )

# This performs Friedman test across the top expert-count x top-k combinations using paired benchmark/task/language rows from the combined benchmark results
def combo_friedman_overall(df, label="overall"):
    # First sort all expert-count x top-k combinations by their mean normalized score
    combo_summary = (
      df.groupby(["component_count", "top_k", "combo"])
      .agg(
          mean_score=("normalized_score", "mean"),
          n_rows=("normalized_score", "count"),
          n_approaches=("kind", "nunique"),
          approaches=("kind", lambda s: ", ".join(sorted(s.unique()))),
      )
      .sort_values("mean_score", ascending=False)
    )

    candidates = combo_summary.head(5).reset_index()["combo"].tolist() # we take the 5 best mean score combos as candidates
    pivot = combo_pivot( # this just builds a paired table for friedman test
      df[df.combo.isin(candidates)],
      ["score_family", "task", "language", "metric"],
    )
    stat, p = friedmanchisquare(*[pivot[c] for c in candidates])
    avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values() # these define the friedman ordering, where lower is better
    top5 = avg_ranks.index.tolist()

    return combo_summary, {
      "analysis": label,
      "pivot": pivot,
      "top5_combos": top5,
      "friedman_p_raw": p,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in top5},
    }


def wilcoxon_adjacent_top3(result):
    """
    here we consider the top3 or top5 results and compare for each 1st vs 2nd, 1st vs 3rd, 1st vs 4th, 1st vs 5th if the difference is significant
    we use wilconxon signed rank + holm correction
    """
    pivot = result["pivot"]
    top5 = result["top5_combos"]

    rows = []
    for a, b in [
      (top5[0], top5[1]),
      (top5[0], top5[2]),
      (top5[0], top5[3]),
      (top5[0], top5[4]),
      (top5[1], top5[2]),
    ]:
      w = wilcoxon(pivot[a], pivot[b], zero_method="pratt", alternative="greater")
      rows.append({
          "analysis": result["analysis"],
          "a": a,
          "b": b,
          "mean_a": pivot[a].mean(),
          "mean_b": pivot[b].mean(),
          "median_delta_a_minus_b": (pivot[a] - pivot[b]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

    out = pd.DataFrame(rows)
    out["p_holm"] = holm(out["p_raw"])
    out["significant"] = out["p_holm"] < 0.05
    return out


def combo_friedman_by_benchmark(df):
    """ basically the same as above but now we look at the benchamrk families (sib, belebele, flores, taxi) separately"""
    rows = []

    for score_family, g in df.groupby("score_family"):
      combo_summary = (
          g.groupby(["component_count", "top_k", "combo"])
          .agg(
              mean_score=("normalized_score", "mean"),
              n_rows=("normalized_score", "count"),
              n_approaches=("kind", "nunique"),
              approaches=("kind", lambda s: ", ".join(sorted(s.unique()))),
          )
          .sort_values("mean_score", ascending=False)
      )

      top3 = combo_summary.head(3).reset_index()["combo"].tolist()

      pivot = combo_pivot(
          g[g.combo.isin(top3)],
          ["task", "language", "metric"],
      )

      stat, p = friedmanchisquare(*[pivot[c] for c in top3])
      avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()

      rows.append({
          "score_family": score_family,
          "winner_combo": avg_ranks.index[0],
          "top3_combos": top3,
          "n_pairs": len(pivot),
          "p_raw": p,
          "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
          **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in top3},
      })

    out = pd.DataFrame(rows)
    out["p_holm"] = holm(out["p_raw"])
    out["significant"] = out["p_holm"] < 0.05
    return out


# only top-k methods
topk_metrics = metrics[metrics.top_k.notna()].copy()

topk_metrics["combo"] = (
  "e" + topk_metrics.component_count.astype(str)
  + " k" + topk_metrics.top_k.astype(int).astype(str)
)

# Overall top3 expert-count x top-k combos
overall_summary, overall_result = combo_friedman_overall(topk_metrics, "overall")
overall_test = pd.DataFrame([{k: v for k, v in overall_result.items() if k != "pivot"}])
overall_test["p_holm"] = holm(overall_test["friedman_p_raw"])
overall_test["significant"] = overall_test["p_holm"] < 0.05

overall_wilcoxon = wilcoxon_adjacent_top3(overall_result)

print("First we look at the best combinations by mean normalized score")
display(overall_summary.head(10))

print("Now we look at the best 5 Friedmann results, ordered by average Friedmann rank")
print("According to Friedmann e32 k4 is best, e16 k1 is 5th best")
display(overall_test)

print("Now we check specifically if the difference between best friedmann rank vs 2nd best Friedmann rank is signifcant simiarl for 1st vs 3rd, 1st vs 4th and 1st vs 5th")
display(overall_wilcoxon)

print("Lastly, we look at what performs best per benchmark family according to Friedman")
benchmark_combo_tests = combo_friedman_by_benchmark(topk_metrics)
display(benchmark_combo_tests)

First we look at the best combinations by mean normalized score


mean_score  n_rows  n_approaches  \
component_count top_k combo                                       
32              2.0   e32 k2     0.914839     126             3   
                16.0  e32 k16    0.908089      84             2   
                8.0   e32 k8     0.907479     126             3   
                4.0   e32 k4     0.907037     126             3   
16              1.0   e16 k1     0.889024     126             3   
32              1.0   e32 k1     0.885014     126             3   
8               2.0   e8 k2      0.879819     126             3   
4               1.0   e4 k1      0.875808     126             3   
                2.0   e4 k2      0.875318     126             3   
16              8.0   e16 k8     0.875146     126             3   

                                         approaches  
component_count top_k combo                          
32              2.0   e32 k2   hmora, moelora, mola  
                16.0  e32 k16           hmora, mola  
                8.0   e32 k8   hmora, moelora, mola  
                4.0   e32 k4   hmora, moelora, mola  
16              1.0   e16 k1   hmora, moelora, mola  
32              1.0   e32 k1   hmora, moelora, mola  
8               2.0   e8 k2    hmora, moelora, mola  
4               1.0   e4 k1    hmora, moelora, mola  
                2.0   e4 k2    hmora, moelora, mola  
16              8.0   e16 k8   hmora, moelora, mola

Now we look at the best 5 Friedmann results, ordered by average Friedmann rank
According to Friedmann e32 k4 is best, e16 k1 is 5th best


,analysis,top5_combos,friedman_p_raw,kendalls_w,avg_rank_e32 k4,avg_rank_e32 k8,avg_rank_e32 k2,avg_rank_e32 k16,avg_rank_e16 k1,p_holm,significant
0,overall,"[e32 k4, e32 k8, e32 k2, e32 k16, e16 k1]",1.115356e-09,0.283643,2.119048,2.464286,2.809524,3.333333,4.27381,1.115356e-09,True


Now we check specifically if the difference between best friedmann rank vs 2nd best Friedmann rank is signifcant simiarl for 1st vs 3rd, 1st vs 4th and 1st vs 5th


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,overall,e32 k4,e32 k8,0.907037,0.907479,0.000158,42,0.336927,0.336927,False
1,overall,e32 k4,e32 k2,0.907037,0.914839,0.000399,42,0.022318,0.089273,False
2,overall,e32 k4,e32 k16,0.907037,0.908089,0.001055,42,0.057922,0.173767,False
3,overall,e32 k4,e16 k1,0.907037,0.889024,0.003703,42,0.002892,0.014459,True
4,overall,e32 k8,e32 k2,0.907479,0.914839,0.000230,42,0.135593,0.271187,False


Lastly, we look at what performs best per benchmark family according to Friedman


,score_family,winner_combo,top3_combos,n_pairs,p_raw,kendalls_w,avg_rank_e32 k8,avg_rank_e32 k4,avg_rank_e32 k1,avg_rank_e32 k2,avg_rank_e4 k2,avg_rank_e16 k4,avg_rank_e16 k8,avg_rank_e32 k16,avg_rank_e16 k1,p_holm,significant
0,Belebele,e32 k8,"[e32 k8, e32 k4, e32 k1]",9,0.169013,0.197531,1.555556,2.000000,2.444444,NaN,NaN,NaN,NaN,NaN,NaN,0.406006,False
1,FLORES en->x,e32 k8,"[e32 k4, e32 k8, e32 k2]",9,0.016390,0.456790,1.555556,1.666667,NaN,2.777778,NaN,NaN,NaN,NaN,NaN,0.065619,False
2,FLORES x->en,e32 k4,"[e32 k4, e32 k2, e32 k8]",9,0.013124,0.481481,2.777778,1.444444,NaN,1.777778,NaN,NaN,NaN,NaN,NaN,0.065619,False
3,SIB200,e4 k2,"[e4 k2, e16 k4, e16 k8]",9,0.169013,0.197531,NaN,NaN,NaN,NaN,1.555556,2.0,2.444444,NaN,NaN,0.406006,False
4,Taxi1500,e32 k2,"[e32 k2, e32 k16, e16 k1]",6,0.135335,0.333333,NaN,NaN,NaN,1.333333,NaN,NaN,NaN,2.333333,2.333333,0.406006,False


### Interpretation
- According to the Friedman ranks, `e32 k4`, `e32 k8`, and `e32 k2` perform best overall
- Wilcoxon signed rank post-hoc tests do not show significant differences between these top combinations


- If we look at mean normalized score, `e32 k2` is highest
- but by Friedman average rank, `e32 k4` is highest, and the top `e32` top-k variants are not significantly different from each other

- if we look at it per benchmark family e32 k4/8/2 are the best e4 k2 wins only on SIB